# 📌 Phase 3 — Détection des Sujets Tendances (Topic Modeling)

**Projet : African Media Intelligence AI**  
**Auteur : Esmel Amari (Phil)**  
**Description :** Ce notebook identifie automatiquement les grandes thématiques présentes dans les articles africains collectés, en utilisant LDA (Latent Dirichlet Allocation) avec scikit-learn comme moteur principal, et une analyse de fréquence de mots-clés.

---

### Méthode
| Outil | Rôle |
|---|---|
| `TfidfVectorizer` | Vectorisation du texte |
| `LatentDirichletAllocation` | Extraction des topics |
| `CountVectorizer` | Analyse de fréquence des mots-clés |
| Mots vides FR+EN | Nettoyage linguistique |


## 0. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import re
import json
import warnings
import logging
from pathlib import Path
from datetime import datetime
from collections import Counter

# NLP / Scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.preprocessing import normalize

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# ── Chemins ────────────────────────────────────────────────────────
BASE_DIR = Path('..').resolve()
PROC_DIR = BASE_DIR / 'data' / 'processed'
PROC_DIR.mkdir(parents=True, exist_ok=True)

# ── Paramètres LDA ────────────────────────────────────────────────
N_TOPICS       = 8   # Nombre de topics à extraire
N_TOP_WORDS    = 10  # Mots-clés par topic
N_TOP_ARTICLES = 5   # Articles représentatifs par topic

print(f"✅ Configuration : {N_TOPICS} topics | {N_TOP_WORDS} mots/topic")

✅ Configuration : 8 topics | 10 mots/topic


## 1. Chargement des Données

In [2]:
# Charger le fichier avec sentiment (sorti de Phase 2)
csv_files = sorted(PROC_DIR.glob('articles_sentiment_*.csv'), reverse=True)

if not csv_files:
    # Fallback : fichier raw
    raw_files = sorted((BASE_DIR / 'data' / 'raw').glob('articles_raw_*.csv'), reverse=True)
    if not raw_files:
        raise FileNotFoundError("Aucun fichier trouvé. Exécutez les notebooks précédents.")
    df = pd.read_csv(raw_files[0], encoding='utf-8-sig')
    print(f"⚠️  Fichier raw utilisé (sentiment absent) : {raw_files[0].name}")
else:
    df = pd.read_csv(csv_files[0], encoding='utf-8-sig')
    print(f"✅ Fichier chargé : {csv_files[0].name}")

print(f"   → {df.shape[0]} articles")

✅ Fichier chargé : articles_sentiment_20260512_2027.csv
   → 121 articles


## 2. Mots Vides (Stop Words) FR + EN

In [3]:
# ── Stop words français étendus ────────────────────────────────────
STOPWORDS_FR = [
    'le','la','les','un','une','des','du','de','en','au','aux','par',
    'pour','dans','sur','avec','sans','sous','entre','vers','chez',
    'et','ou','mais','donc','or','ni','car','que','qui','quoi','dont',
    'où','quand','comment','pourquoi','quel','quelle','quels','quelles',
    'ce','cet','cette','ces','mon','ton','son','ma','ta','sa','notre',
    'votre','leur','mes','tes','ses','nos','vos','leurs',
    'je','tu','il','elle','nous','vous','ils','elles','on','y','en',
    'me','te','se','lui','leur','moi','toi','soi',
    'est','sont','être','avoir','été','fait','faire','dit','dire',
    'plus','moins','très','aussi','même','bien','tout','tous','toute',
    'toutes','autre','autres','comme','ainsi','encore','déjà','alors',
    'selon','dont','lors','après','avant','depuis','pendant',
    'lundi','mardi','mercredi','jeudi','vendredi','samedi','dimanche',
    'janvier','février','mars','avril','mai','juin','juillet','août',
    'septembre','octobre','novembre','décembre',
    'afrique','africain','africains','africaine','africaines'
]

# ── Stop words anglais essentiels ─────────────────────────────────
STOPWORDS_EN = [
    'the','a','an','and','or','but','in','on','at','to','for','of',
    'with','by','from','is','are','was','were','be','been','being',
    'have','has','had','do','does','did','will','would','could','should',
    'may','might','shall','that','this','these','those','it','its',
    'he','she','we','they','his','her','our','their','i','my','you',
    'your','also','said','says','africa','african'
]

ALL_STOPWORDS = list(set(STOPWORDS_FR + STOPWORDS_EN))
print(f"✅ {len(ALL_STOPWORDS)} mots vides chargés (FR + EN)")

✅ 188 mots vides chargés (FR + EN)


## 3. Prétraitement du Texte

In [4]:
def preprocess_text(text: str) -> str:
    """Nettoyage et normalisation du texte pour LDA."""
    if not isinstance(text, str):
        return ""
    # Minuscules
    text = text.lower()
    # Suppression URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    # Suppression ponctuation et chiffres
    text = re.sub(r'[^a-zA-ZÀ-ÿ\s]', ' ', text)
    # Suppression mots très courts (<3 chars)
    text = ' '.join(w for w in text.split() if len(w) >= 3)
    # Suppression espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Appliquer sur la colonne texte principale
df['texte_clean'] = (
    df['titre'].fillna('') + ' ' +
    df['resume'].fillna('') + ' ' +
    df.get('contenu', pd.Series([''] * len(df))).fillna('')
).apply(preprocess_text)

# Filtrer les textes trop courts après nettoyage
df = df[df['texte_clean'].str.split().str.len() >= 10].copy()

print(f"✅ Textes prétraités : {len(df)} articles")
print(f"\nExemple (premier article) :")
print(df['texte_clean'].iloc[0][:300], '...')

✅ Textes prétraités : 121 articles

Exemple (premier article) :
kenya mps probe missing tonnes imported sugar declared unfit for consumption capital nairobi the national assembly committee trade industry and cooperatives has launched investigations into the handling and whereabouts more than metric tonnes imported sugar deemed unfit for human consumption capital ...


## 4. Vectorisation TF-IDF & LDA

In [5]:
# ── Vectorisation ──────────────────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features  = 2000,
    stop_words    = ALL_STOPWORDS,
    min_df        = 2,        # Mot dans au moins 2 articles
    max_df        = 0.90,     # Excluire mots trop fréquents (>90%)
    ngram_range   = (1, 2),   # Unigrammes + bigrammes
    sublinear_tf  = True
)

texts = df['texte_clean'].tolist()
X_tfidf = tfidf.fit_transform(texts)
feature_names = tfidf.get_feature_names_out()

print(f"✅ Matrice TF-IDF : {X_tfidf.shape[0]} articles × {X_tfidf.shape[1]} termes")

# ── Modèle LDA ────────────────────────────────────────────────────
print(f"\n⏳ Entraînement LDA ({N_TOPICS} topics)...")

lda = LatentDirichletAllocation(
    n_components     = N_TOPICS,
    max_iter         = 20,
    learning_method  = 'online',
    learning_offset  = 50.0,
    random_state     = 42,
    n_jobs           = -1
)

doc_topic_matrix = lda.fit_transform(X_tfidf)

print(f"✅ LDA entraîné | Perplexité : {lda.perplexity(X_tfidf):.1f}")

✅ Matrice TF-IDF : 121 articles × 497 termes

⏳ Entraînement LDA (8 topics)...
✅ LDA entraîné | Perplexité : 214511.3


## 5. Labellisation des Topics

In [6]:
def get_top_words(model, feature_names, n_top=10):
    """Extrait les N mots-clés les plus importants par topic."""
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        topics.append(top_words)
    return topics


topic_words = get_top_words(lda, feature_names, N_TOP_WORDS)

# Labels manuels basés sur les mots-clés (à ajuster selon les résultats réels)
# Ces labels sont des suggestions — tu peux les modifier après exécution
TOPIC_LABELS_SUGGESTIONS = [
    "Politique & Gouvernance",
    "Économie & Finance",
    "Sécurité & Conflits",
    "Santé & Développement",
    "Technologie & Innovation",
    "Commerce & Investissement",
    "Énergie & Ressources",
    "Société & Culture"
]

print("=" * 60)
print("TOPICS DÉTECTÉS PAR LDA")
print("=" * 60)

topics_summary = []
for i, words in enumerate(topic_words):
    label = TOPIC_LABELS_SUGGESTIONS[i] if i < len(TOPIC_LABELS_SUGGESTIONS) else f"Topic {i}"
    nb_docs = int((doc_topic_matrix.argmax(axis=1) == i).sum())
    print(f"\n📌 Topic {i} — {label} ({nb_docs} articles)")
    print(f"   Mots-clés : {', '.join(words[:8])}")
    topics_summary.append({
        'topic_id'    : i,
        'label'       : label,
        'mots_cles'   : ', '.join(words),
        'nb_articles' : nb_docs
    })

TOPICS DÉTECTÉS PAR LDA

📌 Topic 0 — Politique & Gouvernance (19 articles)
   Mots-clés : pays, président, plusieurs, manière, bonne, goïta, nord, place

📌 Topic 1 — Économie & Finance (17 articles)
   Mots-clés : fois, cinq, times, continues, east, ghana, ans, attacks

📌 Topic 2 — Sécurité & Conflits (13 articles)
   Mots-clés : nairobi, pays, pouvoir, parti, peut, dynamisme, tensions, trade

📌 Topic 3 — Santé & Développement (13 articles)
   Mots-clés : opposition, demi, rare, marking, office, boualem, president, tchad

📌 Topic 4 — Technologie & Innovation (19 articles)
   Mots-clés : vraiment, tant, drc, groups, chercheurs, mis, pays, intelligence

📌 Topic 5 — Commerce & Investissement (20 articles)
   Mots-clés : country, bamako, grandes, forward, monde, ont, deux, mandat

📌 Topic 6 — Énergie & Ressources (10 articles)
   Mots-clés : abidjan, côte, montant, côte ivoire, service, ivoire, francophone, nigerian

📌 Topic 7 — Société & Culture (10 articles)
   Mots-clés : court, aujourd

## 6. Assignation des Topics aux Articles

In [7]:
# ── Topic dominant par article ─────────────────────────────────────
df['topic_id']    = doc_topic_matrix.argmax(axis=1)
df['topic_score'] = doc_topic_matrix.max(axis=1).round(4)
df['topic_label'] = df['topic_id'].apply(
    lambda i: TOPIC_LABELS_SUGGESTIONS[i]
    if i < len(TOPIC_LABELS_SUGGESTIONS) else f"Topic {i}"
)

# ── Scores pour tous les topics (pour Power BI) ────────────────────
for i in range(N_TOPICS):
    label = TOPIC_LABELS_SUGGESTIONS[i] if i < len(TOPIC_LABELS_SUGGESTIONS) else f"Topic {i}"
    col_name = f"topic_score_{i}_{label.split()[0].lower()}"
    df[col_name] = doc_topic_matrix[:, i].round(4)

print("=== DISTRIBUTION DES TOPICS ===")
topic_dist = df.groupby(['topic_id', 'topic_label']).size().reset_index(name='nb_articles')
topic_dist['pct'] = (topic_dist['nb_articles'] / len(df) * 100).round(1)
print(topic_dist.sort_values('nb_articles', ascending=False).to_string(index=False))

print(f"\n✅ Topics assignés à {len(df)} articles")

=== DISTRIBUTION DES TOPICS ===
 topic_id               topic_label  nb_articles  pct
        5 Commerce & Investissement           20 16.5
        0   Politique & Gouvernance           19 15.7
        4  Technologie & Innovation           19 15.7
        1        Économie & Finance           17 14.0
        2       Sécurité & Conflits           13 10.7
        3     Santé & Développement           13 10.7
        6      Énergie & Ressources           10  8.3
        7         Société & Culture           10  8.3

✅ Topics assignés à 121 articles


## 7. Analyse des Mots-Clés Tendances

In [8]:
# ── Top mots-clés global (CountVectorizer) ─────────────────────────
cv = CountVectorizer(
    max_features = 100,
    stop_words   = ALL_STOPWORDS,
    ngram_range  = (1, 2),
    min_df       = 2
)
X_count = cv.fit_transform(texts)
keywords_freq = pd.DataFrame({
    'mot_cle'   : cv.get_feature_names_out(),
    'frequence' : X_count.toarray().sum(axis=0)
}).sort_values('frequence', ascending=False)

print("=== TOP 20 MOTS-CLÉS GLOBAUX ===")
print(keywords_freq.head(20).to_string(index=False))

# Sauvegarde du vocabulaire tendances
keywords_file = PROC_DIR / 'keywords_trending.csv'
keywords_freq.to_csv(keywords_file, index=False, encoding='utf-8-sig')
print(f"\n✅ Mots-clés tendances sauvegardés : {keywords_file}")

=== TOP 20 MOTS-CLÉS GLOBAUX ===
        mot_cle  frequence
           pays         44
        nairobi         35
         france         31
          monde         28
            ont         27
      président         27
            ans         21
      continent         20
         macron         20
        forward         20
emmanuel macron         19
       emmanuel         19
       français         18
           deux         17
          paris         17
         sommet         17
 sommet forward         16
        capital         16
         contre         15
      president         14

✅ Mots-clés tendances sauvegardés : C:\Users\E682\Desktop\data\processed\keywords_trending.csv


## 8. Sauvegarde

In [9]:
# ── Fichier principal ──────────────────────────────────────────────
output_file = PROC_DIR / f"articles_topics_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"✅ Articles + topics : {output_file}")

# ── Résumé des topics ──────────────────────────────────────────────
topics_df = pd.DataFrame(topics_summary)
topics_file = PROC_DIR / 'topics_summary.csv'
topics_df.to_csv(topics_file, index=False, encoding='utf-8-sig')
print(f"✅ Résumé topics : {topics_file}")

print(f"\n📊 Colonnes ajoutées : topic_id, topic_score, topic_label + scores par topic")

✅ Articles + topics : C:\Users\E682\Desktop\data\processed\articles_topics_20260512_2031.csv
✅ Résumé topics : C:\Users\E682\Desktop\data\processed\topics_summary.csv

📊 Colonnes ajoutées : topic_id, topic_score, topic_label + scores par topic


---

## ✅ Phase 3 Terminée

**Ce qui a été accompli :**
- ✅ Prétraitement linguistique FR + EN
- ✅ Vectorisation TF-IDF (2000 features, bigrammes)
- ✅ LDA entraîné sur les articles — **8 topics détectés**
- ✅ Topic dominant et score assignés à chaque article
- ✅ Top 100 mots-clés tendances exportés

**Prochaine étape :** `04_ner_entities.ipynb` — Extraction des entités nommées (entreprises, pays, organisations)
